Importing UN comtrade data

In [36]:
# We will use a dictionary to organize by year

import pandas as pd

dataframes = {}

dataframes[2012] = pd.read_csv("/Users/kynesantos/Grad School/Trade Networks/Total Trade 2012-2023/(2012) TradeData_2_23_2026_12_33_14.csv", encoding='latin-1', index_col=False)
dataframes[2013] = pd.read_csv("/Users/kynesantos/Grad School/Trade Networks/Total Trade 2012-2023/(2013) TradeData_2_23_2026_12_34_10.csv", encoding='latin-1', index_col=False)
dataframes[2014] = pd.read_csv("/Users/kynesantos/Grad School/Trade Networks/Total Trade 2012-2023/(2014) TradeData_2_23_2026_12_34_43.csv", encoding='latin-1', index_col=False)
dataframes[2015] = pd.read_csv("/Users/kynesantos/Grad School/Trade Networks/Total Trade 2012-2023/(2015) TradeData_2_23_2026_12_35_6.csv", encoding='latin-1', index_col=False)
dataframes[2016] = pd.read_csv("/Users/kynesantos/Grad School/Trade Networks/Total Trade 2012-2023/(2016) TradeData_2_23_2026_12_35_27.csv", encoding='latin-1', index_col=False)
dataframes[2017] = pd.read_csv("/Users/kynesantos/Grad School/Trade Networks/Total Trade 2012-2023/(2017) TradeData_2_23_2026_12_35_56.csv", encoding='latin-1', index_col=False)
dataframes[2018] = pd.read_csv("/Users/kynesantos/Grad School/Trade Networks/Total Trade 2012-2023/(2018) TradeData_2_23_2026_12_36_13.csv", encoding='latin-1', index_col=False)
dataframes[2019] = pd.read_csv("/Users/kynesantos/Grad School/Trade Networks/Total Trade 2012-2023/(2019) TradeData_2_23_2026_12_36_30.csv", encoding='latin-1', index_col=False)
dataframes[2020] = pd.read_csv("/Users/kynesantos/Grad School/Trade Networks/Total Trade 2012-2023/(2020) TradeData_2_23_2026_12_37_18.csv", encoding='latin-1', index_col=False)
dataframes[2021] = pd.read_csv("/Users/kynesantos/Grad School/Trade Networks/Total Trade 2012-2023/(2021) TradeData_2_23_2026_12_37_43.csv", encoding='latin-1', index_col=False)
dataframes[2022] = pd.read_csv("/Users/kynesantos/Grad School/Trade Networks/Total Trade 2012-2023/(2022) TradeData_2_23_2026_12_37_59.csv", encoding='latin-1', index_col=False)
dataframes[2023] = pd.read_csv("/Users/kynesantos/Grad School/Trade Networks/Total Trade 2012-2023/(2023) TradeData_2_23_2026_12_38_21.csv", encoding='latin-1', index_col=False)
dataframes[2024] = pd.read_csv("/Users/kynesantos/Grad School/Trade Networks/Total Trade 2012-2023/(2024) TradeData_2_23_2026_12_38_42.csv", encoding='latin-1', index_col=False)




In [37]:
# The export between A to B is sometimes reported differently by each party. 
# We will opt to believe the importer when the values differ, and save these simplified dataframes under a dictionary all_flows keyed by year.

imports = {}
exports = {}
all_flows = {}

for year in range(2012,2025):

    imports[year] = dataframes[year][dataframes[year]['flowCode'] == 'M'].copy()
    exports[year] = dataframes[year][dataframes[year]['flowCode'] == 'X'].copy()

    imports[year] = imports[year][['partnerISO', 'reporterISO', 'primaryValue']]
    imports[year].columns = ['source', 'target', 'weight']
    imports[year]['priority'] = 0 

    exports[year] = exports[year][['reporterISO', 'partnerISO', 'primaryValue']]
    exports[year].columns = ['source', 'target', 'weight']
    exports[year]['priority'] = 1  

    all_flows[year] = pd.concat([imports[year], exports[year]])
    
    all_flows[year] = all_flows[year].sort_values('priority')
    all_flows[year] = all_flows[year].drop_duplicates(subset=['source', 'target'], keep='first')
    all_flows[year] = all_flows[year].drop(columns='priority')

In [38]:
import networkx as nx 

# Now turn each year's dataframe into a graph using a new set of dictionaries

trade_digraph = {} # digraph
trade_graph = {} # undirected graph
trade_revdigraph = {} # reversed digraph

# Nodes which represent broad geographic regions will be removed
node_removal = ['W00', '_X ', 'S19', 'XX ', 'X2 ', 'E19', 'F19','A79', 'O19','X1 ', '_CI', 'A59']

for year in range(2012,2025):

    G = nx.from_pandas_edgelist(
        all_flows[year],
        source="source",
        target="target",
        edge_attr="weight",
        create_using=nx.DiGraph()
    )
    undirected_G = nx.from_pandas_edgelist(
        all_flows[year],
        source="source",
        target="target",
        edge_attr="weight",
        create_using=nx.Graph()
    )
    G.remove_nodes_from(node_removal)
    G_rev = G.reverse(copy=True)
    undirected_G.remove_nodes_from(node_removal)

    trade_digraph[year] = G
    trade_graph[year] = undirected_G
    trade_revdigraph[year] = G_rev



In [39]:
# Checking the number of nodes and edges in a particular year's graph

print(trade_digraph[2014].number_of_nodes())
print(trade_digraph[2014].number_of_edges())

233
31874


In [42]:
# A function to prune a graph to the top k EXPORT partners (#1 country that one exports to)

def keep_top_k_neighbors(G, k):
    G_pruned = nx.DiGraph()
    G_pruned.add_nodes_from(G.nodes())
    
    for node in G.nodes():
        # Get all edges from this node with weights
        edges = [(node, neighbor, G[node][neighbor]['weight']) 
                 for neighbor in G.successors(node)]
        
        # Sort by weight descending and take top k
        top_edges = sorted(edges, key=lambda x: x[2], reverse=True)[:k]
        
        # Add to pruned graph
        for u, v, w in top_edges:
            G_pruned.add_edge(u, v, primaryValue=w)
    
    return G_pruned

# A function to prune a graph to the top k import partners (#1 country that one imports from)
def keep_top_k_importers(G, k):
    G_pruned = nx.DiGraph()
    G_pruned.add_nodes_from(G.nodes())
    
    for node in G.nodes():
        # Get all edges from this node with weights
        edges = [(node, neighbor, G[neighbor][node]['weight']) 
                 for neighbor in G.predecessors(node)]
        
        # Sort by weight descending and take top k
        top_edges = sorted(edges, key=lambda x: x[2], reverse=True)[:k]
        
        # Add to pruned graph
        for u, v, w in top_edges:
            G_pruned.add_edge(u, v, primaryValue=w)
    
    return G_pruned


In [44]:
# Creating the edge set of the k1 graph, using the top importer edges
trade2014k1_imports = keep_top_k_importers(trade_digraph[2014], k=1)
trade2014k1_importedges = nx.to_pandas_edgelist(trade2014k1_imports)
trade2014k1_importedges.to_csv("/Users/kynesantos/Grad School/Trade Networks/Total Trade 2012-2023/2014 k1 (importers v3).csv", index=False)

In [10]:
# Creating the edge set using only the top exporter edges
trade2014k1 = keep_top_k_neighbors(trade_digraph[2014], k=1)
trade2014k1_edges = nx.to_pandas_edgelist(trade2014k1)
trade2014k1_edges.to_csv("/Users/kynesantos/Grad School/Trade Networks/Total Trade 2012-2023/2014 k1.csv", index=False)


# Testing out the minimum spanning tree algorithm
# Spoiler: it's a fail since it takes the smallest trade relationships instead of the biggest ones, 
# will need to alter the source code to get the maximal spanning tree
mst2014 = nx.minimum_spanning_tree(trade_graph[2014])
mst2014_edges = nx.to_pandas_edgelist(mst2014)
mst2014_edges.to_csv("/Users/kynesantos/Grad School/Trade Networks/Total Trade 2012-2023/2014 MST.csv", index=False)

In [35]:
for year in range(2012,2025):

    trade_digraph[year] = keep_top_k_neighbors(trade_digraph[year], 3)